# Jacobian lens — walkthrough

## 0. Setup

Load a model, load a pre-fitted Jacobian lens from the Hub, apply it to a prompt, and render the interactive slice visualisation.

In [1]:
!git clone https://github.com/anthropics/jacobian-lens.git
%cd jacobian-lens

!pip install -e .
%pip install pandas

c:\Users\jason\CS\jacobian-lens\jacobian-lens


fatal: destination path 'jacobian-lens' already exists and is not an empty directory.


Obtaining file:///C:/Users/jason/CS/jacobian-lens/jacobian-lens
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for jlens (pyproject.toml): started
  Building editable for jlens (pyproject.toml): finished with status 'done'
  Created wheel for jlens: filename=jlens-0.1.0-0.editable-py3-none-any.whl size=8957 sha256=54c8651f531f9856392934ec3ee45cd646a46bb913fd27d97275ef194094aa5f
  Stored in directory: C:\Users\jason\AppData\Local\Temp\pip-ephem-wheel-cache-l3tsx5n9\wheels\3d\07\90\005f77be2ccbe6152c6e28f571c9f23bba104d3e12e9eb

In [2]:
import torch
print(torch.__version__)
print(torch.__file__)

import jlens
jlens.configure_logging()

MODEL_NAME = "Qwen/Qwen3.5-4B"
# MODEL_NAME = "Qwen/Qwen3.6-27B"

LENS_REPO = "neuronpedia/jacobian-lens"
LENS_REVISION = "qwen-n1000"
LENS_FILE = {
    "Qwen/Qwen3.5-4B": "qwen3.5-4b/jlens/Salesforce-wikitext/Qwen3.5-4B_jacobian_lens_n1000.pt",
    "Qwen/Qwen3.6-27B": "qwen3.6-27b/jlens/Salesforce-wikitext/Qwen3.6-27B_jacobian_lens_n1000.pt",
}[MODEL_NAME]

2.14.0+xpu
c:\Users\jason\CS\jacobian-lens\.venv\Lib\site-packages\torch\__init__.py


## 1. Load the model

`jlens.from_hf` wraps an already-loaded HuggingFace model into `LensModel` interface. (Note: .to("xpu") is used to run Qwen model on Intel Arc B580. Nvidia GPU's should use .cuda() instead.)

In [3]:
import torch
import transformers

hf_model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16
).to("xpu")

print(torch.xpu.memory_allocated())

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_NAME)
model = jlens.from_hf(hf_model, tokenizer)
model

c:\Users\jason\CS\jacobian-lens\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 426/426 [00:00<00:00, 3088.32it/s]


8512194560


HFLensModel(Qwen3_5ForCausalLM, n_layers=32, d_model=2560)

## 2. Load a pre-fitted lens

`JacobianLens.from_pretrained` pulls a `.pt` from the Hub (or a local path). The lens holds one `[d_model, d_model]` matrix per layer.

In [4]:
lens = jlens.JacobianLens.from_pretrained(
    LENS_REPO, filename=LENS_FILE, revision=LENS_REVISION
)
lens

Fetching 1 files: 100%|██████████| 1/1 [00:00<00:00, 32.40it/s]


JacobianLens(d_model=2560, n_prompts=1000, source_layers=[0..30] (31 layers))

## 3. Apply: J-lens vs logit lens

`lens.apply(model, prompt, positions=...)` runs one forward pass, transports each layer's residual into the final-layer basis with `J_l`, and decodes through the model's own unembedding. `use_jacobian=False` skips the transport — that's the vanilla logit lens.

Below: a two-hop factual question, read out at the boot token. The J-lens surfaces interpretable tokens at layers where the logit lens is still noise.

In [5]:
prompt = "Fact: The currency used in the country shaped like a boot is"
layers = [
    model.n_layers // 4,
    model.n_layers // 2,
    model.n_layers // 4 * 3,
    model.n_layers - 2,
]

jlens_logits, model_logits, _ = lens.apply(model, prompt, layers=layers, positions=[-2])
logit_lens, _, _ = lens.apply(
    model, prompt, layers=layers, positions=[-2], use_jacobian=False
)


def top5(logits):
    return [tokenizer.decode([t]) for t in logits.topk(5).indices]


for layer in layers:
    print(f"L{layer:>3} logit-lens: {top5(logit_lens[layer][0])}")
    print(f"L{layer:>3} J-lens:     {top5(jlens_logits[layer][0])}")
print(f"model:           {top5(model_logits[0])}")

L  8 logit-lens: ['oman', 'edom', 'ולי', ' Urlaubs', 'GPC']
L  8 J-lens:     [' `', ' boots', ' *', ' `\\', ' heel']
L 16 logit-lens: ['shaw', 'วย', 'amaz', 'REA', '举世']
L 16 J-lens:     ['?', '？', "'?", ' Italy', '____']
L 24 logit-lens: ['的形状', '形状的', '形状', 'shape', '-shaped']
L 24 J-lens:     ['-shaped', ' shape', ' shaped', 'shape', '形状']
L 30 logit-lens: [' is', ' shape', '-shaped', ' shaped', ' heel']
L 30 J-lens:     [' is', ' shape', ' shaped', '-shaped', ' heel']
model:           [' is', ' in', ' on', '.', ' with']


### Cleanup

Working with only 12GB VRAM, so cleaning up after running lens.apply() may be helpful.

In [6]:
import gc
del jlens_logits, model_logits
gc.collect()
torch.xpu.empty_cache()
print(torch.xpu.memory_allocated())

8512195072


## 4. Collecting prompt information

Retrieving top-down-summoning.json and collecting cursory information from its prompts

In [7]:
import json
from pathlib import Path

def load_top_down_summoning_stimuli(data_dir="data/experiments"):
    obj = json.loads((Path(data_dir) / "top-down-summoning.json").read_text(encoding="utf-8"))
    items = obj["items"]
    prompts = [item["stimulus"] for item in items]
    return prompts, items, obj  # keep items around — you'll want key/expected/foil later for scoring

prompts, items, obj = load_top_down_summoning_stimuli()
print(f"{len(prompts)} prompts")
for p in prompts:
    print(len(p.split()), "words —", p)

7 prompts
27 words — She had organised the samples by shade and labelled each one before showing him the options. He recognised the effort, but red had never been his favourite
27 words — So yeah, we'd been hanging out at Jake's place all afternoon, just chilling, when his mom totally freaked out about the mess. Honestly it wasn't even that
25 words — She sealed the final box and stacked it by the door as the light faded. Then she sat down at the empty table and carefully
19 words — The committee had argued through lunch without reaching agreement. When the chair finally called for a vote, the members
18 words — The committee debated for three hours before reaching a verdict. In the end, they declared the proposal utterly
18 words — The committee debated for three hours before reaching a verdict. In the end, they declared the proposal utterly
30 words — The letter arrived on a grey morning, and she read it twice before setting it down. The house felt emptier than it had in years, and e

In [8]:
for item in items:
    ids = model.encode(item["stimulus"], max_length=512)
    print(item["key"], ids.shape[-1], "tokens")

spelling 29 tokens
register 35 tokens
tense 26 tokens
number 21 tokens
pos 20 tokens
pos_tense 20 tokens
tone 34 tokens


## 5. Running lens.apply() on the first example

The first prompt in this json is "She had organised the samples by shade and labelled each one before showing him the options. He recognised the effort, but red had never been his favourite"

Then the model is asked 1. "What word do you think comes next? Answer in one word."
and 2. "Where do you think the author of this passage is from? Answer in one word."

Due to the text using british english, we expect the answer to 1. to be "colour", and the answer to 2. to be "britain". If one response occurs significantly more often than the other, it implies that it is judging the "british-ness" in the text separately for each question, rather than having an internal concept that applies to both. 

In [12]:
item = items[0]  # "spelling"
stimulus = item["stimulus"]
q1 = obj["q1"]  # shared question, from the top-level JSON key
q2 = item["q2"]

# Q1: plain continuation — model just keeps writing
prompt_q1 = f"{stimulus}\n\n{q1}"

# Q2: passage + a separate question turn
prompt_q2 = f"{stimulus}\n\n{q2}"

print("Q1 prompt:", repr(prompt_q1))
print("Q2 prompt:", repr(prompt_q2))

Q1 prompt: 'She had organised the samples by shade and labelled each one before showing him the options. He recognised the effort, but red had never been his favourite\n\nWhat word do you think comes next? Answer in one word.'
Q2 prompt: 'She had organised the samples by shade and labelled each one before showing him the options. He recognised the effort, but red had never been his favourite\n\nWhere do you think the author of this passage is from? Answer in one word.'


In [23]:
import torch
import pandas as pd
from jlens.fitting import valid_position_mask

def run_prompt_top10(prompt, model, tokenizer, lens, top_k=10):
    input_ids = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)["input_ids"][0]
    seq_len = len(input_ids)

    valid_mask = valid_position_mask(seq_len)
    positions = [p for p in range(seq_len) if valid_mask[p]]
    layers = list(lens.source_layers)

    rows = []
    for layer in layers:
        for pos in positions:
            try:
                lens_logits, model_logits, _ = lens.apply(
                    model, prompt, layers=[layer], positions=[pos]
                )
                logits = lens_logits[layer][0]        # [vocab_size] — confirmed shape, no ambiguity
                final_logits = model_logits[0]         # same shape, for rank-vs-final

                top_vals, top_ids = logits.topk(top_k)
                top1_id = top_ids[0].item()
                rank_vs_final = (final_logits > final_logits[top1_id]).sum().item() + 1

                rows.append({
                    "layer": layer,
                    "position": pos,
                    "source_token": tokenizer.decode([int(input_ids[pos].item())]),
                    "top1_token": tokenizer.decode([top1_id]),
                    "rank_vs_final": rank_vs_final,
                    "top10_tokens": [tokenizer.decode([t.item()]) for t in top_ids],
                    "top10_logits": top_vals.tolist(),
                })

            except torch.OutOfMemoryError:
                print(f"OOM at layer={layer}, position={pos}. Skipping.")
                rows.append({"layer": layer, "position": pos, "top1_token": None,
                              "rank_vs_final": None, "top10_tokens": None, "top10_logits": None})

            finally:
                for name in ["lens_logits", "model_logits", "logits", "final_logits"]:
                    if name in locals():
                        del locals()[name]
                gc.collect()
                torch.xpu.empty_cache()   # xpu, not cuda, for your B580

    return pd.DataFrame(rows)

In [24]:
stimulus = items[0]["stimulus"]  # "spelling"
df_spelling = run_prompt_top10(stimulus, model, tokenizer, lens, top_k=10)

grid_top1 = df_spelling.pivot(index="layer", columns="position", values="top1_token")
print(grid_top1)

position             16        17           18            19       20  \
layer                                                                   
0                     .        $\   recognised             ,       ``   
1                     ,         �   recognised          \n\n       ``   
2         <|endoftext|>         ­   recognised             ,       ``   
3         <|endoftext|>         ­   recognised          \n\n       ``   
4                     .        ……   recognised          \n\n       ''   
5                     .         .   recognised          \n\n       ``   
6                     .         ­   recognised            \n       ``   
7                     .        ……   recognised          \n\n      ,''   
8                     .        ……   recognised        colour      ,''   
9                     .        ……   recognised        colour      ,''   
10                    .        ……   recognised        images   effort   
11                    .        ……   recognised   ph

In [25]:
grid_top1 = df_spelling.pivot(index="layer", columns="position", values="top1_token")
grid_top1

position,16,17,18,19,20,21,22,23,24,25,26,27
layer,,,,,,,,,,,,
0,.,$\,recognised,",",``,",",ting,s,--,theless,---,ا
1,",",�,recognised,\n\n,``,",",...,s,--,``,``,\n
2,<|endoftext|>,­,recognised,",",``,.,",",s,–,ing,­,","
3,<|endoftext|>,­,recognised,\n\n,``,.,...,s,--,�,----,'
4,.,……,recognised,\n\n,'',.,.,s,–,________,­,'
5,.,.,recognised,\n\n,``,.,",",s,--,________,----,'
6,.,­,recognised,\n,``,.,.,s,--,________,----,'
7,.,……,recognised,\n\n,",''",.,.,s,--,________,_____,‘
8,.,……,recognised,colour,",''",.,.,s,--,________,----,­


## 6. Rendering a slice page

In [26]:
import gzip
import json

from jlens.vis import build_page, compute_slice, notebook_iframe

# Reuse the same gloss dict from earlier in the walkthrough (for CJK vocab tokens)
gloss = {
    int(k): v for k, v in json.load(gzip.open("assets/qwen_gloss.json.gz")).items()
}

stimulus = items[0]["stimulus"]  # "spelling" item

slice_data = compute_slice(
    model,
    lens,
    stimulus,
    layer_stride=2,
    mask_display=True,
)
page, _, _ = build_page(
    slice_data,
    stimulus,
    title="top-down-summoning: spelling",
    description=items[0]["q2"],  # or any description you'd like
    alt_token=gloss,
)
notebook_iframe(page)

## 7. Processing full file and summarizing results

In [ ]:
print(f"total items to process: {len(items)}")

all_rows = []

for item in items:  # or however many prompts you're pulling from across your experiment files
    key = item.get("key", item.get("stimulus", "")[:20])
    prompt = item["stimulus"]
    try:
        df_prompt = run_prompt_top10(prompt, model, tokenizer, lens, top_k=10)  # or the compute_slice-based version, once that's working
        df_prompt["prompt_key"] = key
        all_rows.append(df_prompt)
        print(f"{key}: done")
    except RuntimeError as e:
        print(f"{key}: stopped — {e}")
        break
    finally:
        gc.collect()
        torch.xpu.empty_cache()

    # you can run this after EVERY prompt, not just at the very end —
    # useful to check the summary is sensible before committing more compute
    full_df = pd.concat(all_rows, ignore_index=True)
    summary = full_df.groupby("layer").agg(
        mean_rank=("rank_vs_final", "mean"),
        median_rank=("rank_vs_final", "median"),
        top1_matches_final=("rank_vs_final", lambda s: (s == 1).mean()),
        n_samples=("rank_vs_final", "count"),
    ).reset_index()

print(f"prompts successfully processed: {len(all_rows)}")
print(f"unique prompt_keys in full_df: {full_df['prompt_key'].nunique()}")